In [1]:
!pip install flash-attn sympy math_verify pylatexenc


[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: python3.12 -m pip install --upgrade pip


In [2]:
from vllm import LLM, SamplingParams

# Create an LLM.
llm = LLM(model='Qwen/Qwen2.5-Math-1.5B')

INFO 08-25 22:23:59 [__init__.py:244] Automatically detected platform rocm.
INFO 08-25 22:24:18 [config.py:853] This model supports multiple tasks: {'classify', 'reward', 'score', 'generate', 'embed'}. Defaulting to 'generate'.
INFO 08-25 22:24:18 [config.py:1467] Using max model len 4096
INFO 08-25 22:24:26 [config.py:2267] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 08-25 22:24:26 [config.py:4566] full_cuda_graph is not supported with cascade attention. Disabling cascade attention.
WARNING 08-25 22:24:27 [utils.py:2613] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reason: CUDA is initialized
INFO 08-25 22:24:30 [__init__.py:244] Automatically detected platform rocm.
INFO 08-25 22:24:40 [core.py:459] Waiting for init message from front-end.
INFO 08-25 22:24:40 [core.py:69] Initializing a V1 LLM en

Loading safetensors checkpoint shards:   0% Completed | 0/1 [00:00<?, ?it/s]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.08s/it]
Loading safetensors checkpoint shards: 100% Completed | 1/1 [00:01<00:00,  1.08s/it]



INFO 08-25 22:24:42 [default_loader.py:272] Loading weights took 1.17 seconds
INFO 08-25 22:24:42 [gpu_model_runner.py:1782] Model loading took 3.0352 GiB and 1.438540 seconds
INFO 08-25 22:24:47 [backends.py:509] Using cache directory: /root/.cache/vllm/torch_compile_cache/d38fb0b184/rank_0_0/backbone for vLLM's torch.compile
INFO 08-25 22:24:47 [backends.py:520] Dynamo bytecode transform time: 4.59 s
INFO 08-25 22:24:49 [backends.py:155] Directly load the compiled graph(s) for shape None from the cache, took 0.473 s
INFO 08-25 22:24:50 [monitor.py:34] torch.compile takes 4.59 s in total
INFO 08-25 22:25:04 [gpu_worker.py:232] Available KV cache memory: 163.12 GiB
INFO 08-25 22:25:04 [kv_cache_utils.py:716] GPU KV cache size: 6,108,624 tokens
INFO 08-25 22:25:04 [kv_cache_utils.py:720] Maximum concurrency for 4,096 tokens per request: 1491.36x
INFO 08-25 22:25:04 [rocm.py:224] Using Triton Attention backend on V1 engine.


Capturing CUDA graphs: 100%|██████████| 67/67 [00:09<00:00,  7.22it/s]


INFO 08-25 22:25:13 [gpu_model_runner.py:2306] Graph capturing finished in 9 secs, took 0.27 GiB
INFO 08-25 22:25:13 [core.py:172] init engine (profile, create kv cache, warmup model) took 31.20 seconds


In [3]:
question = "Simplify $(3-i)(6+2i)$."

# Sample prompts.
prompts = [
    f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>""",
]

# Create a sampling params object, stopping generation on newline.
sampling_params = SamplingParams(
    temperature=1.0, top_p=1.0, max_tokens=1024, stop=["\n"]
)

sampling_params.stop = ["</answer>"]
sampling_params.include_stop_str_in_output = True

# Generate texts from the prompts. The output is a list of RequestOutput objects
# that contain the prompt, generated text, and other information.
outputs = llm.generate(prompts, sampling_params)

# Print the outputs.
for output in outputs:
    prompt = output.prompt
    generated_text = output.outputs[0].text
    print(f"Prompt: {prompt!r}")
    print(f"Generated text: {generated_text!r}")

Adding requests:   0%|          | 0/1 [00:00<?, ?it/s]

Processed prompts:   0% 0/1 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: Simplify $(3-i)(6+2i)$.\nAssistant: <think>'
Generated text: ' We can simplify the expression by applying the distributive law.\n$$(3-i)(6+2i)\\\\ = 3\\cdot 6+3\\cdot 2i-(6\\cdot i)+(2i\\cdot i)\\\\ =18+6i-6i+2i^2.$$\nWe know $i^2 = -1$, so...\n<end think>\nAssistant: <answer> We can simplify our expression to $18 + 2i^2$.\nTherefore our final answer is $16 + 2i$'


In [4]:
from drgrpo_grader import r1_zero_reward_fn

In [5]:
r1_zero_reward_fn(generated_text, '20')

{'format_reward': 0.0, 'answer_reward': 0.0, 'reward': 0.0}

In [8]:
from typing import Callable

def evaluate_vllm(
  vllm_model: LLM,
  reward_fn: Callable[[str, str], dict[str, float]],
  prompts: list[str],
  answers: list[str],
  eval_sampling_params: SamplingParams
) -> tuple[float, float, float]:
  """
  Evaluate a language model on a list of prompts,
  compute evaluation metrics, and serialize results to disk.
  """
  outputs = vllm_model.generate(prompts, eval_sampling_params)
  format_reward = 0.
  answer_reward = 0.
  reward = 0.
  for index, output in enumerate(outputs):
    prompt = output.prompt
    generated_text = output.outputs[0].text
    rewards = reward_fn(generated_text, answers[index])
    format_reward += rewards['format_reward']
    answer_reward += rewards['answer_reward']
    reward += rewards['reward']
    if rewards['reward'] == 1.0:
        print(f"Prompt: {prompt!r}")
        print(f"Generated text: {generated_text!r}")
        print(f"Answer: {answers[index]}")
        print(f"Reward: {rewards}")
        print('*****************')

  return format_reward, answer_reward, reward

In [ ]:
import pandas as pd
from tqdm import tqdm


df = pd.read_parquet("math_12k.parquet")

total_format_reward = 0.
total_answer_reward = 0.
total_reward = 0.
prompts = []
answers = []

for index, row in tqdm(df.iterrows()):
    question = row['problem']
    answer = row['solution']
    prompt = f"""A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.
User: {question}
Assistant: <think>"""
    prompts.append(prompt)
    answers.append(answer)

    if (index + 1) % 80 == 0:
      format_reward, answer_reward, reward = evaluate_vllm(llm, r1_zero_reward_fn, prompts, answers, sampling_params)
      total_format_reward += format_reward
      total_answer_reward += answer_reward
      total_reward += reward
      prompts = []
      answers = []

print(f"Total format reward: {total_format_reward}")
print(f"Total answer reward: {total_answer_reward}")
print(f"Total reward: {total_reward}")

0it [00:00, ?it/s]

Adding requests:   0%|          | 0/80 [00:00<?, ?it/s]

Processed prompts:   0% 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

80it [00:05, 14.44it/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: A freight train travels 1 mile in 1 minute 30 seconds. At this rate, how many miles will the train travel in 1 hour?\nAssistant: <think>'
Generated text: ' The train travels 1 mile in 90 seconds, i.e., 3/2 minutes. Hence, in 1 hour, the train will travel 2/3 mile per minute*60 minutes=40 miles. </think> <answer> 40 miles </answer>'
Answer: 40
Reward: {'format_reward': 1.0, 'answer_reward': 1.0, 'reward': 1.0}
*****************
Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about

...

Processed prompts:   0% 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

1360it [01:39, 14.02it/s]

Prompt: 'A conversation between User and Assistant. The User asks a question, and the Assistant solves it. The Assistant first thinks about the reasoning process in the mind and then provides the User with the answer. The reasoning process is enclosed within <think> </think> and answer is enclosed within <answer> </answer> tags, respectively, i.e., <think> reasoning process here </think> <answer> answer here </answer>.\nUser: The number of inches in the perimeter of a square is equal to the number of square inches in its area. Find the length, in inches, of a side of the square.\nAssistant: <think>'
Generated text: "Let 's' be the length of a side of the square. The perimeter is 4s, and the area is s^2. We are given that the perimeter equals the area, so we can set up the equation 4s = s^2. We need to solve this equation for 's'.</think>\n<s>Next, we solve the equation 4s = s^2. By rearranging the terms, we get s^2 - 4s = 0. Factoring out 's', we have s(s - 4) = 0. This gives us two po

Adding requests:   0%|          | 0/80 [00:00<?, ?it/s]

Processed prompts:   0% 0/80 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]